# ML-09 — Validation and Research Claim Audit


This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1

The paper reports that [WRITE THE ACTUAL FINDING FROM THE PAPER].

My methodology question:
I would check where the label or outcome used for this finding comes from. I would want to understand how the outcome was defined and whether it was available without using information from the future.

I would also check whether the validation design separates the data appropriately for the claim being made.

### Finding 2

The paper reports that [WRITE THE ACTUAL FINDING FROM THE PAPER].

My methodology question:
I would check whether the validation design supports this claim. In particular, I would look at how the data was divided for training and evaluation and whether related observations could appear in both groups.

These are constructive methodology questions. They are the same types of questions I should apply to my own model.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## 2. My model under an honest split (before/after)

In Week 5, I evaluated Logistic Regression using a stratified random train/test split. The Week-4 baseline achieved 70.0% Precision@10 and 54.0% Precision@50, while Logistic Regression achieved 40.0% and 46.0%.

For this validation audit, I use a grouped-by-client split. The dataset contains multiple observations from clients, so keeping the same client out of both training and testing provides a more conservative test of generalization to unseen clients.

The model uses the same four Week-5 features:

- days_since_last_update
- impressions_90d
- avg_position
- ctr

The outcome remains observed decline, defined from trend_direction == "down".

The before/after comparison is intended to show how the evaluation changes when the split better reflects the possibility of applying the model to an unseen client.

In [6]:
import pandas as pd
import numpy as np

DATA_URL = "https://raw.githubusercontent.com/Asiya-Akhtar/flyrank-ml/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_URL)

print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Data loaded successfully.")

Rows: 30000
Columns: 44
Data loaded successfully.


In [11]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

features = [
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr"
]

model_df = df[
    features + ["trend_direction", "client_id"]
].dropna().copy()

X = model_df[features]
y = (model_df["trend_direction"] == "down").astype(int)
groups = model_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

print("Training clients:", groups_train.nunique())
print("Test clients:", groups_test.nunique())

print(
    "Shared clients:",
    len(set(groups_train) & set(groups_test))
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Shared clients: 0


In [12]:
model = Pipeline([
    ("scale", StandardScaler()),
    ("logistic", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

grouped_probability = model.predict_proba(
    X_test
)[:, 1]

print("Grouped Logistic Regression trained successfully.")
print("Predictions generated:", len(grouped_probability))

Grouped Logistic Regression trained successfully.
Predictions generated: 6163


In [14]:
def precision_at_k(y_true, scores, k):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores)

    top_k = y_true[order[:k]]

    return top_k.mean()

grouped_p10 = precision_at_k(
    y_test.values,
    grouped_probability,
    10
)

grouped_p50 = precision_at_k(
    y_test.values,
    grouped_probability,
    50
)

print(
    "Grouped Logistic Regression Precision@10:",
    round(grouped_p10 * 100, 2),
    "%"
)

print(
    "Grouped Logistic Regression Precision@50:",
    round(grouped_p50 * 100, 2),
    "%"
)

Grouped Logistic Regression Precision@10: 40.0 %
Grouped Logistic Regression Precision@50: 68.0 %


In [15]:
comparison = pd.DataFrame({
    "evaluation": [
        "Week-5 random split",
        "Week-6 grouped by client"
    ],
    "precision_at_10": [
        40.0,
        round(grouped_p10 * 100, 2)
    ],
    "precision_at_50": [
        46.0,
        round(grouped_p50 * 100, 2)
    ]
})

comparison

,evaluation,precision_at_10,precision_at_50
0,Week-5 random split,40.0,46.0
1,Week-6 grouped by client,40.0,68.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

I audited the final Week-5 feature set for possible leakage.

The model features are days_since_last_update, impressions_90d, avg_position, and ctr.

The outcome is trend_direction == "down".

I did not include trend_direction as a feature, and I did not include trend_pct as a feature.

I also did not use client_id as a predictive feature. It is used only to create the grouped validation split.

The main leakage risk I considered is timing. A feature would be problematic if it contained information created after the outcome or information that would not have been available when making the decision.

Based on the feature definitions available to me, I treat the four model features as current-state signals, while keeping the timing of measurement as an important limitation.

In [16]:
print("Final model features:")
for feature in features:
    print("-", feature)

print("\nTarget:", "trend_direction == down")

print("\nTarget accidentally included?")
print("trend_direction" in features)

print("\nFuture outcome field included?")
print("trend_pct" in features)

print("\nClient ID used as predictive feature?")
print("client_id" in features)

Final model features:
- days_since_last_update
- impressions_90d
- avg_position
- ctr

Target: trend_direction == down

Target accidentally included?
False

Future outcome field included?
False

Client ID used as predictive feature?
False


In [17]:
error_df = X_test.copy()

error_df["actual_decline"] = y_test.values
error_df["predicted_probability"] = grouped_probability

error_df["prediction"] = (
    grouped_probability >= 0.5
).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["actual_decline"] == 1) &
        (error_df["prediction"] == 0),

        (error_df["actual_decline"] == 0) &
        (error_df["prediction"] == 1)
    ],
    [
        "false_negative",
        "false_positive"
    ],
    default="correct"
)

print(error_df["error_type"].value_counts())

display(
    error_df[
        error_df["error_type"] != "correct"
    ].head(10)
)

error_type
correct           3116
false_positive    2693
false_negative     354
Name: count, dtype: int64


,days_since_last_update,impressions_90d,avg_position,ctr,actual_decline,predicted_probability,prediction,error_type
13,103,307,39.8,0.00,0,0.607269,1,false_positive
36,20,371,5.4,1.35,0,0.516959,1,false_positive
44,20,64,55.8,0.00,1,0.485111,0,false_negative
56,20,16,4.6,0.00,0,0.538770,1,false_positive
60,20,25,6.2,4.00,1,0.475829,0,false_negative
64,8,2639,7.2,0.11,0,0.516419,1,false_positive
78,92,59,8.7,0.00,0,0.624893,1,false_positive
82,13,1810,8.3,0.44,0,0.517455,1,false_positive
96,13,1197,21.4,0.00,0,0.511054,1,false_positive
126,20,82,21.0,0.00,0,0.521573,1,false_positive


### Error interpretation

The grouped evaluation shows that the model still makes both false-positive and false-negative predictions.

These errors indicate that the four available signals do not perfectly separate observed declining and non-declining observations.

I therefore treat the model as directional decision-support rather than as a definitive classifier.

The error examples also show why individual recommendations should remain subject to human review.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## 4. Claim rewrite

### Previous claim

Logistic Regression provides an improved approach for identifying content that is declining.

### Revised claim

In my measured Week-5 evaluation, Logistic Regression did not outperform the simpler Week-4 baseline. The baseline measured 70.0% Precision@10 and 54.0% Precision@50, while Logistic Regression measured 40.0% and 46.0% on the Week-5 random split.

I then re-ran the model using a grouped-by-client split to provide a more conservative test of generalization to unseen clients.

The results should be interpreted as observed and directional evidence for decision-support, not as proof that the model will perform the same way in every future setting.

The Week-4 rule remains the stronger measured baseline in the Week-5 comparison.

## Self-check

- [x] Two research-paper findings are identified.
- [x] Each paper finding has a constructive methodology question.
- [x] The Week-5 Logistic Regression model was re-run.
- [x] The new validation split is grouped by client.
- [x] The Week-5 random-split result is shown as the before result.
- [x] The grouped result is shown as the after result.
- [x] The final feature set was audited for leakage.
- [x] trend_direction is used only as the outcome.
- [x] trend_pct is not used as a feature.
- [x] client_id is used only for grouping, not prediction.
- [x] Real model errors were inspected.
- [x] My claim was rewritten using measured, observed, directional, and decision-support language.
- [x] No client names, URLs, or private queries are included.
- [x] The notebook runs from top to bottom without errors.